In [1]:
## Importing Libraries and Paths

import os
import cv2
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm.notebook import tqdm

from sklearn.preprocessing import StandardScaler

from skimage.feature import hog
from skimage.feature import local_binary_pattern
from skimage.feature import graycomatrix
from skimage.feature import graycoprops

print("Libraries Loaded Successfully.")

# Paths

ROOT_DIR = Path.cwd().parent

DATASET_DIR = ROOT_DIR / "datasets" / "processed_dataset"

TRAIN_DIR = DATASET_DIR / "images" / "train"
VAL_DIR = DATASET_DIR / "images" / "val"
TEST_DIR = DATASET_DIR / "images" / "test"

TRAIN_LABELS = DATASET_DIR / "labels" / "train"
VAL_LABELS = DATASET_DIR / "labels" / "val"
TEST_LABELS = DATASET_DIR / "labels" / "test"

FEATURE_DIR = DATASET_DIR / "ml_features"
FEATURE_DIR.mkdir(exist_ok=True)

print("Directories Loaded Successfully.")

Libraries Loaded Successfully.
Directories Loaded Successfully.


In [2]:
## Loading Dataset

train_images = sorted(TRAIN_DIR.glob("*"))
val_images = sorted(VAL_DIR.glob("*"))
test_images = sorted(TEST_DIR.glob("*"))

print("=" * 50)

print(f"Training Images   : {len(train_images)}")
print(f"Validation Images : {len(val_images)}")
print(f"Testing Images    : {len(test_images)}")

print("=" * 50)

Training Images   : 3948
Validation Images : 846
Testing Images    : 847


In [3]:
## Extracting Features

def extract_hog(image):
    features = hog(image, orientations=8, pixels_per_cell=(16, 16), cells_per_block=(2, 2), block_norm="L2-Hys", visualize=False, feature_vector=True)

    return features[:1000]

def extract_lbp(image):
    radius = 3
    points = radius * 8

    lbp = local_binary_pattern(image, points, radius, method="uniform")
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, points + 3), range=(0, points + 2))

    hist = hist.astype("float")
    hist /= (hist.sum() + 1e-6)

    return hist

def extract_color(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    hist = cv2.calcHist([hsv], [0, 1, 2], None, [4, 4, 4], [0, 180, 0, 256, 0, 256])
    hist = cv2.normalize(hist, hist).flatten()

    return hist

def extract_hu(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    moments = cv2.moments(gray)

    hu = cv2.HuMoments(moments).flatten()
    hu = np.sign(hu) * np.log1p(np.abs(hu))

    return hu

def extract_glcm(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    glcm = graycomatrix(gray, distances=[1], angles=[0], symmetric=True, normed=True)

    contrast = graycoprops(glcm, "contrast")[0, 0]
    correlation = graycoprops(glcm, "correlation")[0, 0]
    energy = graycoprops(glcm, "energy")[0, 0]
    homogeneity = graycoprops(glcm, "homogeneity")[0, 0]

    return np.array([contrast, correlation, energy, homogeneity])

In [4]:
## Master Feature Extractor

IMAGE_SIZE = (128, 128)

def extract_features(image_path):
    image = cv2.imread(str(image_path))
    image = cv2.resize(image, IMAGE_SIZE)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    hog_features = extract_hog(gray)
    lbp_features = extract_lbp(gray)
    color_features = extract_color(image)
    hu_features = extract_hu(image)
    glcm_features = extract_glcm(image)

    feature_vector = np.concatenate([hog_features, lbp_features, color_features, hu_features, glcm_features])

    return feature_vector

In [5]:
## Reading YOLO Labels and Extracting Class

def read_label(label_path):
    with open(label_path, "r") as f:
        lines = f.readlines()

    class_id = int(lines[0].split()[0])

    return class_id


print("Feature extraction pipeline ready.")

Feature extraction pipeline ready.


In [6]:
## Extracting Training Features

X_train = []
y_train = []

for image_path in tqdm(train_images, desc="Training Features"):
    label_path = TRAIN_LABELS / (image_path.stem + ".txt")
    features = extract_features(image_path)
    label = read_label(label_path)

    X_train.append(features)
    y_train.append(label)

X_train = np.array(X_train)
y_train = np.array(y_train)

print("Training Feature Matrix :", X_train.shape)

Training Features:   0%|          | 0/3948 [00:00<?, ?it/s]

Training Feature Matrix : (3948, 1101)


In [7]:
## Extracting Validation and Testing Features

def process_dataset(images, labels, dataset_name):
    X = []
    y = []

    for image_path in tqdm(images, desc=dataset_name):
        label_path = labels / (image_path.stem + ".txt")
        features = extract_features(image_path)
        label = read_label(label_path)

        X.append(features)
        y.append(label)

    return np.array(X), np.array(y)


X_val, y_val = process_dataset(val_images, VAL_LABELS, "Validation Features")
X_test, y_test = process_dataset(test_images, TEST_LABELS, "Testing Features")

print("Validation :", X_val.shape)
print("Testing    :", X_test.shape)

Validation Features:   0%|          | 0/846 [00:00<?, ?it/s]

Testing Features:   0%|          | 0/847 [00:00<?, ?it/s]

Validation : (846, 1101)
Testing    : (847, 1101)


In [8]:
## Scaling Features

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("Feature Scaling Completed.")

Feature Scaling Completed.


In [9]:
## Saving Feature Datasets

pd.DataFrame(X_train).assign(Label=y_train).to_csv(FEATURE_DIR / "train_features.csv", index=False)
pd.DataFrame(X_val).assign(Label=y_val).to_csv(FEATURE_DIR / "val_features.csv", index=False)
pd.DataFrame(X_test).assign(Label=y_test).to_csv(FEATURE_DIR / "test_features.csv", index=False)

joblib.dump(scaler, FEATURE_DIR / "scaler.pkl")

with open(FEATURE_DIR / "feature_names.txt", "w") as f:
    f.write(f"Total Features : {X_train.shape[1]}")

print("Feature datasets saved successfully.")

Feature datasets saved successfully.


In [10]:
## Final Verification

print("=" * 50)
print("FEATURE ENGINEERING COMPLETED")
print("=" * 50)

print(f"Training Samples    : {len(X_train)}")
print(f"Validation Samples  : {len(X_val)}")
print(f"Testing Samples     : {len(X_test)}")

print()

print(f"Features/Image      : {X_train.shape[1]}")

print()

print("Generated Files:")

print("- train_features.csv")
print("- val_features.csv")
print("- test_features.csv")
print("- scaler.pkl")
print("- feature_names.txt")

print()

print(f"Output Directory : {FEATURE_DIR}")

print("=" * 50)
print("Notebook 2 Completed Successfully")
print("=" * 50)

FEATURE ENGINEERING COMPLETED
Training Samples    : 3948
Validation Samples  : 846
Testing Samples     : 847

Features/Image      : 1101

Generated Files:
- train_features.csv
- val_features.csv
- test_features.csv
- scaler.pkl
- feature_names.txt

Output Directory : c:\Users\Akshat\Desktop\Over_Here\Data Vidwan Internship\AgroWeedGuard (Capstone Project)\datasets\processed_dataset\ml_features
Notebook 2 Completed Successfully
